<a href="https://colab.research.google.com/github/shizoda/education/blob/main/machine_learning/MNIST_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3層ニューラルネットワークを用いたMNIST分類

手書き数字データセット MNIST を対象とし、3層ニューラルネットワークによる分類と学習を学びます。

- 入力は 28x28 画素の1チャネル画像における濃度値の1次元ベクトルです。
- 3層構造（入力層・隠れ層・出力層）のニューラルネットワークを可視化します。
- 出力は各クラスに属する確率を表す10個の数値であり、その総和は 1 になります。
- 正解の One-hot 表現は、ネットワークが出力すべき理想的な値です。
- 出力と正解 One-hot 表現を比較して誤差を計算します。
- 誤差を減少させる過程を学習と呼び、ネットワークの更新には誤差逆伝播法を用います。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from tqdm import tqdm
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score, log_loss
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
import warnings

warnings.filterwarnings("ignore")

# MNISTデータセットの取得と縮小サンプリング
mnist = fetch_openml("mnist_784", version=1, as_frame=False, parser="auto")
X_all, y_all = mnist.data, mnist.target.astype(int)

# 実行時間を短縮するために一部のデータをサンプリング
X_sample, _, y_sample, _ = train_test_split(
    X_all, y_all, train_size=5000, random_state=42, stratify=y_all
)

# 濃度値(0-255)を [0.0, 1.0] に正規化
X_norm = X_sample / 255.0

# 訓練データと検証データに分割
X_train, X_val, y_train, y_val = train_test_split(
    X_norm, y_sample, test_size=1000, random_state=42, stratify=y_sample
)

## 画像の濃度値とベクトル化

MNISTの画像は縦 28 画素、横 28 画素の1チャネル構成です。ニューラルネットワークの入力とするため、二次元の画素配列を長さ 784 の1次元ベクトルへ変換します。

ベクトルの各要素はそれぞれの画素の濃度値に対応します。

In [ ]:
# 1サンプルの画像(28x28)と1次元ベクトル(784,)の可視化
sample_idx = 0
image_2d = X_train[sample_idx].reshape(28, 28)
vector_1d = X_train[sample_idx]*255

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.imshow(image_2d, cmap="gray")
plt.title("28x28 Image (1 channel)")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.plot(vector_1d)
plt.title("784-dimensional Input Vector")
plt.xlabel("Pixel Index")
plt.ylabel("Value")
plt.ylim(0, 255)

plt.tight_layout()
plt.show()

## 課題

- 右のデータは左の画像から変換されたものですが、何を描いたものでしょうか？

- このあと右のデータを何に使いますか？

## 3層ニューラルネットワークの可視化

ネットワークは 入力層、 隠れ層、 出力層 の3層で構成されます。

- 入力層：画像の画素数と同じ 784 ノード
- 隠れ層：特徴の組み合わせを処理する中間層（本コードでは可視化と計算の都合上 32 ノードとしています）
- 出力層：分類するクラス数と同じ 10 ノード（0 から 9 の数字に対応）

In [ ]:
# 正解ラベルを One-hot 表現へ変換
encoder = OneHotEncoder(sparse_output=False)
y_train_onehot = encoder.fit_transform(y_train.reshape(-1, 1))
y_val_onehot = encoder.transform(y_val.reshape(-1, 1))

def draw_mlp_network(input_nodes, hidden_nodes, output_nodes):
    G = nx.DiGraph()
    layers = [input_nodes, hidden_nodes, output_nodes]
    layer_names = ["Input Layer\n(784)", "Hidden Layer\n(32)", "Output Layer\n(10)"]

    # 可視化のために描画ノード数を間引き
    display_counts = [15, 8, 10]

    pos = {}
    node_colors = []

    for i, (layer_size, disp_count) in enumerate(zip(layers, display_counts)):
        y_offsets = np.linspace(-4, 4, disp_count)
        for j in range(disp_count):
            node_id = f"L{i}_{j}"
            G.add_node(node_id)
            pos[node_id] = (i * 3, y_offsets[j])

            if i == 0:
                node_colors.append("lightblue")
            elif i == 1:
                node_colors.append("lightgreen")
            else:
                node_colors.append("salmon")

    for j1 in range(display_counts[0]):
        for j2 in range(display_counts[1]):
            G.add_edge(f"L0_{j1}", f"L1_{j2}")

    for j1 in range(display_counts[1]):
        for j2 in range(display_counts[2]):
            G.add_edge(f"L1_{j1}", f"L2_{j2}")

    plt.figure(figsize=(8, 6))
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=250)
    nx.draw_networkx_edges(G, pos, alpha=0.3, arrows=False)

    for i, name in enumerate(layer_names):
        plt.text(i * 3, 5, name, ha="center", va="center", fontsize=11, fontweight="bold")

    plt.axis("off")
    plt.show()

draw_mlp_network(784, 32, 10)

## 正解 One-hot と損失関数の仕組み

分類タスクにおいて、数値ラベル（例：数値の 3）は **One-hot 表現** （例：`[0, 0, 0, 1, 0, 0, 0, 0, 0, 0]`）に変換して取り扱います。これは各クラスであるべき確率を表す理想的な目標値です。

ネットワークの予測確率と、正解 One-hot とのズレを評価するために **クロスエントロピー損失** を用います。予測確率が正解 One-hot に近づくほど、損失の値は 0 に収束します。

In [ ]:
class MNISTSampleEvaluator:
    """
    サンプルの画像視認、ベクトル確認、出力確率とOne-hotの比較、損失評価を行うクラス
    """
    def __init__(self, encoder):
        self.encoder = encoder

    def evaluate(self, model, X_data, y_data, sample_idx):
        x_vector = X_data[sample_idx]
        true_label = y_data[sample_idx]
        true_onehot = self.encoder.transform([[true_label]])[0]

        # 1. 画像とベクトルの可視化とprint出力
        print("-" * 60)
        print(f"サンプルインデックス: {sample_idx} (正解ラベル: {true_label})")
        print("784次元ベクトルの要素数:", len(x_vector))
        print("-" * 60)

        # 2. 確率予測と総和の確認
        x_input = x_vector.reshape(1, -1)
        proba_output = model.predict_proba(x_input)[0]
        proba_sum = np.sum(proba_output)

        print("モデルの出力確率ベクトル:", np.round(proba_output, 4))
        print(f"出力確率の合計和: {proba_sum:.4f}")
        print("正解 One-hot ベクトル :", true_onehot.astype(int))

        # 3. クロスエントロピー損失の計算
        loss_val = log_loss([true_onehot], [proba_output])
        print(f"このサンプルのクロスエントロピー損失: {loss_val:.4f}")
        print("-" * 60)

        # 4. グラフ可視化
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

        # 28x28画像の描画
        ax1.imshow(x_vector.reshape(28, 28), cmap="gray")
        ax1.set_title(f"Input Image (True: {true_label})")
        ax1.axis("off")

        # 出力確率とOne-hotの比較棒グラフ
        classes = np.arange(10)
        ax2.bar(classes - 0.2, proba_output, width=0.4, label="Model Output", color="salmon")
        ax2.bar(classes + 0.2, true_onehot, width=0.4, label="Ideal One-hot", color="lightblue")
        ax2.set_xticks(classes)
        ax2.set_xlabel("Class Index (Digit)")
        ax2.set_ylabel("Probability / Target")
        ax2.set_ylim(0, 1.1)
        ax2.set_title(f"Output vs One-hot (Loss: {loss_val:.4f})")
        ax2.legend()

        plt.tight_layout()
        plt.show()

evaluator = MNISTSampleEvaluator(encoder)

## 学習前の出力確認

未学習の状態でバリデーションデータから 1 サンプルを取得し、先述の評価クラスを呼び出します。

学習前は内部の重みが初期値であるため、どの数字かも判断できず、確率はすべてのクラスでおよそ 10 分の 1 に均等分散し、損失の値は高くなります。

In [ ]:
from sklearn.utils import check_random_state
from sklearn.preprocessing import LabelBinarizer

def initialize_untrained_mlp():
    """
    MLPClassifier を未学習（ランダム初期パラメータ）状態で初期化し、
    推論および誤差計算を行えるように設定する関数
    """
    model = MLPClassifier(
        hidden_layer_sizes=(32,),
        max_iter=1,
        warm_start=True,
        random_state=42,
        batch_size=64,
        solver="sgd",
        learning_rate_init=0.05
    )

    # scikit-learnの内部推論に必要な属性を明示的に設定
    model.n_layers_ = 3
    model.n_outputs_ = 10
    model.out_activation_ = "softmax"
    model.classes_ = np.arange(10)

    lb = LabelBinarizer()
    lb.fit(model.classes_)
    model._label_binarizer = lb

    # 乱数生成器の割り当てと各層における重み・バイアスの生成
    model._random_state = check_random_state(model.random_state)
    layer_units = [784, 32, 10]
    model.coefs_ = []
    model.intercepts_ = []

    for i in range(len(layer_units) - 1):
        coef, intercept = model._init_coef(layer_units[i], layer_units[i+1], np.float64)
        model.coefs_.append(coef)
        model.intercepts_.append(intercept)

    return model

# モデルを未学習状態で作成
mlp = initialize_untrained_mlp()

# 学習前のサンプル出力評価
sample_index = 0
print("【学習前の出力確認】")
evaluator.evaluate(mlp, X_val, y_val, sample_idx=sample_index)

## 課題

- 左の画像に関して、右のグラフにおいて、赤は＿＿＿＿＿＿＿・青は＿＿＿＿＿＿を表します。

- これから学習を行うことで、変わっていくのはどちら？ どのように変わっていく？

## 正解 One-hot と誤差逆伝播法

モデルの出力は、入力画像がクラス $0$ から $9$ に属するそれぞれの確率を表す 10 個の数値です。確率の総和は必ず 1 となります。

正解ラベルは「One-hot 表現」に変換します。これは正解のクラスの位置が 1 ・それ以外が 0 のベクトルで、ネットワークの理想的な出力に相当します。

学習手順は以下の通りです。

1. 出力確率ベクトルと正解 One-hot ベクトルの比較による誤差の計算
2. 誤差を減らす方向へパラメータを微修正する **誤差逆伝播法** （バックプロパゲーション）の適用
3. これらを繰り返すことによる損失関数の最小化

In [ ]:

# 3層ニューラルネットワークの構築 (隠れ層1層 = 3層構造)
mlp = MLPClassifier(
    hidden_layer_sizes=(32,),
    max_iter=1,
    warm_start=True,
    random_state=42,
    batch_size=64,
    solver="sgd",
    learning_rate_init=0.1
)

max_epochs = 40
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

with tqdm(total=max_epochs, leave=False) as pbar:
    for epoch in range(max_epochs):
        mlp.fit(X_train, y_train)

        # 予測確率の算出
        y_train_proba = mlp.predict_proba(X_train)
        y_val_proba = mlp.predict_proba(X_val)

        # クロスエントロピー誤差(Loss)の計算
        train_loss = log_loss(y_train_onehot, y_train_proba)
        val_loss = log_loss(y_val_onehot, y_val_proba)

        # 正解率(Accuracy)の計算
        train_acc = accuracy_score(y_train, mlp.predict(X_train))
        val_acc = accuracy_score(y_val, mlp.predict(X_val))

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accuracies.append(train_acc)
        val_accuracies.append(val_acc)

        pbar.set_description(f"Epoch {epoch+1} - Loss: {train_loss:.4f}, Acc: {train_acc:.4f}")
        pbar.update(1)

# 学習経過の可視化
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Cross Entropy Loss")
plt.title("Loss over Epochs (Error Minimization)")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_accuracies, label="Train Accuracy")
plt.plot(val_accuracies, label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy over Epochs")
plt.legend()

plt.tight_layout()
plt.show()

## 学習後の出力確認

学習完了後に、学習前と同じ検証データサンプルを用いて再度評価クラスを呼び出します。正解 One-hot ベクトルの位置の確率が上昇し、損失関数が変化していることが確認できます。

In [ ]:
print("【学習後の出力確認】")
evaluator.evaluate(mlp, X_val, y_val, sample_idx=sample_index)

## 課題

- ネットワークの出力は赤のグラフです。7 のところで高い値、それ以外で低い値になっているはずですが、それはどういう意味ですか？

- 赤（ネットワーク）を青（正解 One-hot）に近づけるのが ＿＿＿＿＿ です。